# Uji Coba Folium dengan Sampel Acak Dataset GPS

Sel ini memuat `pandas` dan `folium` untuk mengambil beberapa ratus data acak dari berkas `.parquet` kamu, lalu menampilkannya sebagai _marker_ di atas peta interaktif.

In [1]:
import pandas as pd
import folium

# File path yang akan digunakan, misalnya dari bulan April 2022
# Disarankan mengganti pada dataset lain bila diperlukan.
file_path = 'DataGPS_parquet/gps_ring_road_utara.parquet'

print(f"Membaca dataset: {file_path} ...")
# Jika data sangat besar, load hanya kolom latitude dan longitude jika sudah tahu nama aslinya
df = pd.read_parquet(file_path)

# Mengambil 500 sampel titik acak
sample_df = df.sample(n=500, random_state=42)

# Mengidentifikasi kolom latitude dan longitude dari list kolom yang ada
# (Disesuaikan dengan format di dataset kamu: misalnya 'latitude', 'lat', 'y', dsb.)
lat_col = next((col for col in sample_df.columns if 'lat' in col.lower() or col == 'y'), None)
lon_col = next((col for col in sample_df.columns if 'lon' in col.lower() or col == 'x'), None)

if not lat_col or not lon_col:
    print(f"Kolom koordinat tidak ditemukan. Kolom yang tersedia: {list(sample_df.columns)}")
else:
    print(f"Menggunakan kolom: {lat_col} untuk lat dan {lon_col} untuk lon.")

    # Titik tengah peta didapatkan dari rata-rata titik sampel
    center_lat = sample_df[lat_col].mean()
    center_lon = sample_df[lon_col].mean()
    
    # Inisialisasi peta
    m = folium.Map(location=[center_lat, center_lon], zoom_start=12)

    # Tambahkan marker / circle untuk setiap data di atas peta
    for idx, row in sample_df.iterrows():
        folium.CircleMarker(
            location=[row[lat_col], row[lon_col]],
            radius=3,
            color='blue',
            fill=True,
            fill_color='blue',
            fill_opacity=0.6,
            popup=f"ID/Info: {idx}"
        ).add_to(m)

# Panggil "m" saja di akhir agar peta tampil di jupyter
m

Membaca dataset: DataGPS_parquet/gps_ring_road_utara.parquet ...
Menggunakan kolom: latitude untuk lat dan longitude untuk lon.


# Alternatif dengan KeplerGL

Karena peta dasar OpenStreetMap pada Folium memblokir akses jika *referrer* tidak sesuai (Error `403r Access blocked`), kita bisa mempertimbangkan untuk menggunakan layanan peta lain atau alat seperti **KeplerGL** yang berbasis Mapbox / deck.gl.

In [ ]:
import pandas as pd
from keplergl import KeplerGl

# Sama seperti sebelumnya, baca dataset
file_path = 'DataGPS_parquet/2022_04_April.parquet'
df = pd.read_parquet(file_path)

# Ambil sampel (keplergl lebih sanggup menerima banyak data dibanding folium)
sample_df = df.sample(n=1000, random_state=42)

# Buat map
map_1 = KeplerGl(height=600)
map_1.add_data(data=sample_df, name='gps_data')

# Tampilkan
map_1

# Peta Jalan Tol di Yogyakarta

Sel berikut mendemonstrasikan cara membuat garis (PolyLine) pada Folium untuk menggambarkan rute jalan tol yang ada atau sedang dibangun di Jogja (misalnya Tol Solo-Jogja dan Tol Jogja-Bawen).

*(Titik koordinat di bawah ini adalah perkiraan rute untuk kebutuhan demonstrasi penggambaran jalan)*. Untuk analisis presisi, sebaiknya import `GeoJSON` trase jalan tol resmi menggunakan `folium.GeoJson` atau `geopandas`)*.

In [2]:
import folium

# Titik tengah peta wilayah DI Yogyakarta
jogja_center = [-7.7956, 110.3695]

# Membuat peta dengan tiles CartoDB positron agar terhindar dari Error 403 OSM
m_tol = folium.Map(location=jogja_center, zoom_start=11, tiles='CartoDB positron')

# === Perkiraan titik koordinat trase Tol Solo - Yogyakarta - YIA (Seksi 1) ===
# Mulai dari perbatasan Prambanan/Klaten menuju Purwomartani dan Ringroad Utara
tol_solo_jogja = [
    [-7.7289, 110.5186], # Area Klaten/Prambanan
    [-7.7551, 110.4578], # Kalasan
    [-7.7659, 110.4225], # Purwomartani
    [-7.7423, 110.3860], # Area Maguwoharjo / Ringroad Utara
]

# === Perkiraan titik koordinat trase Tol Yogyakarta - Bawen (Seksi 1 Jogja-Banyurejo) ===
# Mulai dari Tirtoadi/Sleman menuju utara (Tempel)
tol_jogja_bawen = [
    [-7.7225, 110.3385], # Mlati / Tirtoadi (Junction)
    [-7.6978, 110.3235], # Seyegan
    [-7.6536, 110.2858], # Banyurejo / Tempel (Perbatasan Magelang)
]

# Menambahkan garis Tol Solo-Jogja (Warna Merah)
folium.PolyLine(
    locations=tol_solo_jogja,
    color='red',
    weight=5,
    opacity=0.8,
    tooltip='Tol Solo - Yogyakarta (Perkiraan Trase)'
).add_to(m_tol)

# Menambahkan garis Tol Jogja-Bawen (Warna Biru)
folium.PolyLine(
    locations=tol_jogja_bawen,
    color='blue',
    weight=5,
    opacity=0.8,
    tooltip='Tol Yogyakarta - Bawen (Perkiraan Trase)'
).add_to(m_tol)

# Tambahkan penanda untuk mempermudah identifikasi
folium.Marker([-7.7659, 110.4225], popup='Gerbang Tol Purwomartani (Estimasi)', icon=folium.Icon(color='red')).add_to(m_tol)
folium.Marker([-7.7225, 110.3385], popup='Junction Tirtoadi (Estimasi)', icon=folium.Icon(color='blue')).add_to(m_tol)

# Menampilkan peta
m_tol

# Visualisasi Jaringan Jalan OSMnx Menggunakan Folium

Di bawah ini adalah contoh blok kode untuk mengunduh jaringan jalan (graph) dari OpenStreetMap menggunakan `osmnx` dan menampilkannya di atas peta interaktif `folium`.

Untuk mempercepat waktu unduh pada demonstrasi ini, digunakan *radius point* sekitar 1 km di pusat kota (sekitar Tugu Jogja) daripada mengunduh seluruh kota Yogyakarta sekaligus.

In [5]:
import osmnx as ox
import folium

# ============================================================
# Gunakan nama wilayah administratif DI Yogyakarta
# ============================================================
place_name = "Daerah Istimewa Yogyakarta, Indonesia"

print(f"Mengunduh graph OSMnx untuk wilayah: {place_name} ...")

# Custom filter: hanya motorway, trunk, dan primary
custom_filter = '["highway"~"motorway|trunk|primary"]'

# Unduh graf berdasarkan batas administratif (bukan radius titik)
G = ox.graph_from_place(
    place_name,
    custom_filter=custom_filter,
    retain_all=True
)

print(f"Graph berhasil diunduh. Total node: {len(G.nodes)}, Total edge: {len(G.edges)}")

# Ubah ke GeoDataFrame
nodes, edges = ox.graph_to_gdfs(G)

# Hitung center peta dari bounding box semua node
center_lat = nodes.geometry.y.mean()
center_lon = nodes.geometry.x.mean()
center_point = (center_lat, center_lon)

print(f"Center peta otomatis: {center_point}")

# ============================================================
# Buat peta Folium
# ============================================================
m_osmnx = folium.Map(
    location=center_point,
    zoom_start=11,          # zoom out lebih jauh untuk DIY
    tiles='CartoDB positron'
)

# Warna berdasarkan tipe jalan
highway_style = {
    'motorway': {'color': '#e74c3c', 'weight': 4},   # merah
    'trunk':    {'color': '#e67e22', 'weight': 3},   # oranye
    'primary':  {'color': '#0066cc', 'weight': 2},   # biru
}

print("Menambahkan jalur ke peta Folium...")

for _, row in edges.iterrows():
    if row.geometry.geom_type == 'LineString':
        coords = [(lat, lon) for lon, lat in row.geometry.coords]

        # Ambil tipe highway; bisa berupa list, ambil elemen pertama
        hw = row.get('highway', 'primary')
        if isinstance(hw, list):
            hw = hw[0]

        style = highway_style.get(hw, {'color': '#999999', 'weight': 2})
        road_name = row.get('name', 'Jalan Tanpa Nama')
        if isinstance(road_name, list):
            road_name = road_name[0]

        folium.PolyLine(
            locations=coords,
            color=style['color'],
            weight=style['weight'],
            opacity=0.8,
            tooltip=f"[{hw.upper()}] {road_name}"
        ).add_to(m_osmnx)

# ============================================================
# Tambahkan legenda
# ============================================================
legend_html = """
<div style="position: fixed; bottom: 30px; left: 30px; z-index: 1000;
     background: white; padding: 12px 16px; border-radius: 8px;
     box-shadow: 0 2px 8px rgba(0,0,0,0.3); font-family: Arial; font-size: 13px;">
  <b>Tipe Jalan</b><br>
  <span style="color:#e74c3c;">&#9644;</span> Motorway<br>
  <span style="color:#e67e22;">&#9644;</span> Trunk<br>
  <span style="color:#0066cc;">&#9644;</span> Primary
</div>
"""
m_osmnx.get_root().html.add_child(folium.Element(legend_html))

# Simpan
output_file = 'peta_osmnx_DIY.html'
m_osmnx.save(output_file)
print(f"Peta berhasil disimpan: {output_file}")

Mengunduh graph OSMnx untuk wilayah: Daerah Istimewa Yogyakarta, Indonesia ...
Graph berhasil diunduh. Total node: 1357, Total edge: 2692
Center peta otomatis: (np.float64(-7.8153731106116435), np.float64(110.38181065305821))
Menambahkan jalur ke peta Folium...
Peta berhasil disimpan: peta_osmnx_DIY.html


In [3]:
import osmnx as ox
import matplotlib.pyplot as plt
import geopandas as gpd

# 1. Ambil jaringan jalan Ring Road Utara, Yogyakarta
place_name = "Yogyakarta, Indonesia"

# Ambil semua jalan di area Yogyakarta
G = ox.graph_from_place(place_name, network_type="drive")

# 2. Filter hanya jalan bernama "Ring Road Utara"
edges = ox.graph_to_gdfs(G, nodes=False)

ring_road_utara = edges[
    edges["name"].apply(
        lambda x: "Ring Road Utara" in x if isinstance(x, str)
        else any("Ring Road Utara" in n for n in x) if isinstance(x, list)
        else False
    )
]

print(f"Jumlah segmen Ring Road Utara: {len(ring_road_utara)}")
print(ring_road_utara[["name", "highway", "length", "geometry"]].head(10))

# 3. Plot
fig, ax = plt.subplots(figsize=(12, 8))

# Background: semua jalan di Yogyakarta (abu-abu tipis)
edges.plot(ax=ax, color="lightgray", linewidth=0.5, alpha=0.5)

# Ring Road Utara (merah tebal)
ring_road_utara.plot(ax=ax, color="red", linewidth=3, label="Ring Road Utara")

ax.set_title("Ring Road Utara – Yogyakarta", fontsize=16, fontweight="bold")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.legend()
plt.tight_layout()
plt.savefig("ring_road_utara.png", dpi=150)
plt.show()

# 4. Simpan ke GeoJSON (opsional)
ring_road_utara.to_file("ring_road_utara.geojson", driver="GeoJSON")
print("Disimpan ke ring_road_utara.geojson")

KeyboardInterrupt: 

In [2]:
import folium

# Inisialisasi peta folium (koordinat sekitar Yogyakarta)
m = folium.Map(location=[-7.7598, 110.3814], zoom_start=13)

# File path untuk geojson yang berisi Ring Road Utara
geojson_path = 'cache/ring_road_utara_edges.geojson'

# Tambahkan layer GeoJson ke dalam peta
folium.GeoJson(
    geojson_path,
    name='Ring Road Utara',
    style_function=lambda feature: {
        'color': 'blue',
        'weight': 3,
        'opacity': 0.8
    }
).add_to(m)

# Tambahkan kontrol layer untuk bisa menonaktifkan/mengaktifkan layer
folium.LayerControl().add_to(m)

# Tampilkan peta
m